# Tega: walk through the linked financial model

The history is audited consolidated FY2024/FY2025 data in INR million. Every forecast is illustrative. This is a historical FY2025 case, not a current Tega valuation.

Run the cells in order. No additional Python packages are needed for the calculations; this notebook itself needs a notebook environment such as VS Code with Jupyter support. Precomputed outputs are included for reading on GitHub.

## 1. Find the repository and load the two separate inputs

Reported figures live in the statements file. Analyst choices live in the assumptions file.

In [1]:
import json
import sys
from pathlib import Path

root = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "examples/tega_fy2025_reported_statements.json").is_file()
    ),
    None,
)
if root is None:
    raise RuntimeError("Open the notebook from the extracted repository folder.")
sys.path.insert(0, str(root / "src"))

from equity_analytics.forecasting import build_forecast, load_json

case = load_json(root / "examples/tega_fy2025_reported_statements.json")
assumptions = load_json(root / "examples/tega_fy2025_forecast_assumptions.json")
result = build_forecast(case, assumptions)
print("Reported company:", case["company"])
print("Information cutoff:", case["as_of"])
print("Historical checks passed:", len(result["historical_checks"]))
print("Balanced forecast years:", len(result["years"]))

Reported company: Tega Industries Limited
Information cutoff: 2025-08-26
Historical checks passed: 64
Balanced forecast years: 5


## 2. Inspect the actual historical statements

These values come from the source report. Cash capex is the cash outflow for capital assets, expressed here as a positive amount.

In [2]:
for year in case["annuals"]:
    print(
        "FY",
        year["fiscal_year"],
        {
            "revenue": year["income"]["revenue"],
            "net_income": year["income"]["net_income"],
            "assets": year["reported_totals"]["assets"],
            "operating_cash_flow": year["cash_flow"]["operating_total"],
            "cash_capex": -year["cash_flow"]["investing"]["capital_asset_purchases"],
        },
    )

FY 2024 {'revenue': 14927.14, 'net_income': 1938.57, 'assets': 18901.39, 'operating_cash_flow': 2521.42, 'cash_capex': 554.12}
FY 2025 {'revenue': 16386.51, 'net_income': 2001.2, 'assets': 20952.02, 'operating_cash_flow': 1950.3, 'cash_capex': 1701.8}


## 3. See how debt can change without cash borrowing

New leases and foreign-exchange movements affect the liability. Interest accrual and payment are separate. Positive movements increase debt; negative movements reduce it.

In [3]:
print(json.dumps(case["fy2025_reconciliations"]["lease_liability"], indent=2))

{
  "opening": 648.71,
  "new_leases_noncash": 173.19,
  "cash_principal_movement": -183.42,
  "interest_accrual": 60.27,
  "interest_paid": -60.27,
  "exchange": 43.31,
  "other": -4.59,
  "closing": 677.2
}


## 4. Follow the first forecast year

`engine.py` calculates revenue, working capital, asset depreciation, interest and tax, then rolls cash and retained earnings. The balance sheet is summed independently.

In [4]:
first = result["years"][0]
print("FY", first["fiscal_year"], "ILLUSTRATIVE FORECAST")
print(json.dumps(first["income"], indent=2))
print("Cash:", round(first["cash_flow"]["closing_cash"], 2))
print("Assets:", round(first["balance_sheet"]["total_assets"], 2))
print(
    "Liabilities plus equity:",
    round(
        first["balance_sheet"]["total_liabilities"]
        + first["balance_sheet"]["total_equity"],
        2,
    ),
)
print("Balance residual:", round(first["checks"]["balance_sheet_residual"], 8))

FY 2026 ILLUSTRATIVE FORECAST
{
  "revenue": 18025.161,
  "ebitda": 3695.158005,
  "depreciation_amortisation": 954.0284428666668,
  "ebit": 2741.129562133333,
  "joint_venture_profit": 44.71,
  "finance_cost": 297.87435,
  "profit_before_tax": 2487.965212133333,
  "taxable_profit": 2443.255212133333,
  "tax": 610.8138030333332,
  "net_income": 1877.1514090999997
}
Cash: 500.0
Assets: 21754.86
Liabilities plus equity: 21754.86
Balance residual: 0.0


## 5. Inspect depreciation and debt schedules

The opening plant group has an assumed remaining life. New assets receive half-year depreciation. These lives are analyst estimates, not reported remaining lives.

In [5]:
selected = [
    r
    for r in first["asset_schedule"]
    if r["name"] in {"ppe_plant_and_wear_parts", "ppe_2026"}
]
print(json.dumps(selected, indent=2))
print(json.dumps(first["debt_schedule"], indent=2))

[
  {
    "name": "ppe_plant_and_wear_parts",
    "account": "ppe",
    "opening": 1641.66,
    "additions": 0.0,
    "depreciation_amortisation": 547.22,
    "closing": 1094.44
  },
  {
    "name": "ppe_2026",
    "account": "ppe",
    "opening": 0.0,
    "additions": 1461.6257888,
    "depreciation_amortisation": 91.3516118,
    "closing": 1370.274177
  }
]
[
  {
    "pool": "term_debt",
    "opening": 1190.62,
    "cash_draw": 0,
    "noncash_new_leases": 0,
    "principal_repayment": 199.97,
    "interest_expense_and_cash_paid": 101.2027,
    "closing": 990.6499999999999
  },
  {
    "pool": "revolver",
    "opening": 1428.67,
    "cash_draw": 0,
    "noncash_new_leases": 0,
    "principal_repayment": 892.4580146868059,
    "interest_expense_and_cash_paid": 135.72365000000002,
    "closing": 536.2119853131942
  },
  {
    "pool": "leases",
    "opening": 677.2,
    "cash_draw": 0,
    "noncash_new_leases": 175,
    "principal_repayment": 179.34,
    "interest_expense_and_cash_paid"

## 6. Change collection days and recalculate

Ten extra days of customer credit ties up more cash in receivables. The first-year operating profit is unchanged; operating cash flow and FCFF fall. The model adjusts financing under the stated cash and facility rules.

In [6]:
from copy import deepcopy

slower = deepcopy(assumptions)
slower["receivable_days"][0] += 10
changed = build_forecast(case, slower)["years"][0]
print(
    "Additional receivables:",
    round(
        changed["balance_sheet"]["assets"]["receivables"]
        - first["balance_sheet"]["assets"]["receivables"],
        2,
    ),
)
print(
    "Change in operating cash flow:",
    round(changed["cash_flow"]["operating"] - first["cash_flow"]["operating"], 2),
)
print("Change in FCFF:", round(changed["fcff"] - first["fcff"], 2))
print("Balance residual:", round(changed["checks"]["balance_sheet_residual"], 8))

Additional receivables: 493.84
Change in operating cash flow: -493.84
Change in FCFF: -493.84
Balance residual: 0.0


## 7. Read the DCF with its assumptions

The valuation uses linked operating FCFF, an explicit terminal reinvestment assumption, and a bridge that includes leases as debt. WACC, terminal growth, ROIC and nonoperating values are illustrative. There is no current share-price comparison.

In [7]:
dcf = result["dcf"]
print("Historical FY2025 illustration; information cutoff", result["as_of"])
print("WACC:", assumptions["dcf"]["wacc"])
print("Terminal growth:", assumptions["dcf"]["terminal_growth"])
print("Terminal ROIC:", assumptions["dcf"]["terminal_roic"])
if dcf["available"]:
    print("Illustrative INR per share:", round(dcf["value_per_share_inr"], 2))
    print(
        "PV terminal value / enterprise value:",
        round(100 * dcf["terminal_share_of_enterprise_value"], 1),
        "%",
    )
else:
    print(dcf["reason"])

Historical FY2025 illustration; information cutoff 2025-08-26
WACC: 0.12
Terminal growth: 0.04
Terminal ROIC: 0.15
Illustrative INR per share: 320.11
PV terminal value / enterprise value: 74.8 %


Read `docs/TEGA_MODEL_METHODS.md` for each convention and limitation. To create fresh reports, run `run_tega_model.py` or double-click `Run_Tega_Model.bat`. Changing inputs in this notebook does not overwrite the JSON input files.